# Fabric Defect Classification — Baseline CNN

**Notebook 03** of the fabric defect classification pipeline.

This notebook trains a **simple convolutional neural network built from scratch** and measures how well it performs.

### A note on the word "baseline"

The word gets used for two different things in this project, so it is worth separating them:

| Name used here | What it is |
|---|---|
| **Majority-class reference** | Not a model at all — just the score you get by always answering "defect free". Calculated from the class counts in Notebook 01: 60.4% accuracy, 0.084 macro F1. It is the floor any real model must clear. |
| **Baseline CNN** *(this notebook)* | A real model, trained from scratch on these 2,737 images alone. It is a *baseline* in the sense that it is the thing ResNet18 must beat. |
| **Main model** *(Notebook 04)* | ResNet18 with transfer learning. |

**Why build the middle one at all?** The project's claim is that transfer learning helps. To show that, a good ResNet18 score is not enough on its own — maybe the task was easy, and any network would have managed it. Training an ordinary CNN from scratch answers that question directly: whatever it achieves is what this dataset yields *without* outside knowledge, so anything ResNet18 gains beyond it comes from the pretraining.

### How we measure success

Notebook 01 showed accuracy is misleading here — a model that learned nothing still scores 60.4%, because "defect free" is 60% of the data. So we report:

- **Macro F1** as the headline, which treats all nine classes as equally important
- **Recall for each class separately**, so the rare defects cannot hide behind the common ones

## 1. Setup

**What:** Load the libraries, find the cleaned data from Notebook 02, and pick a device to train on.

**Why:** Notebook 02 saved the processed images along with `manifest.csv` (which lists each image's class and fold) and `class_weights.json`. Where those files live depends on how you are running — inside the same Kaggle session, from an attached dataset, or on your own machine — so we check each possible location in turn.

Setting a fixed random seed matters more than it might seem. Training involves randomness at every step: how weights start, how images are shuffled, which augmentations are applied. Fixing the seed means re-running this notebook gives the same numbers, so any improvement we see later comes from the change we made rather than luck.

In [ ]:
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             recall_score)
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

# ---------------- Configuration ----------------
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 25
LR = 1e-3
SEED = 42
NUM_WORKERS = 2
N_FOLDS = 5
# -----------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Look for the processed data produced by Notebook 02
CANDIDATES = [
    Path("/content/data/processed"),          # Colab local disk
    Path("/kaggle/working/processed"),        # Kaggle same session
]
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    CANDIDATES += sorted(kaggle_input.glob("*/processed"))   # attached notebook output
    CANDIDATES += sorted(kaggle_input.glob("*"))             # dataset attached at its root
CANDIDATES.append(Path("data/processed"))                    # local machine

DATA_DIR = next((c for c in CANDIDATES if (c / "manifest.csv").exists()), None)

# Results go into their own folder per model, so Notebook 04's ResNet18
# results sit alongside these rather than mixed in with them:
#   outputs/
#   |-- baseline_cnn/
#   \-- resnet18/
MODEL_NAME = "baseline_cnn"

if Path("/kaggle/working").exists():
    OUTPUTS_ROOT = Path("/kaggle/working/outputs")
elif Path("/content").exists():
    OUTPUTS_ROOT = Path("/content/outputs")
else:
    OUTPUTS_ROOT = Path("outputs")

RESULTS_DIR = OUTPUTS_ROOT / MODEL_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device  :", device)
print("Results :", RESULTS_DIR.resolve())
print("Data    :", DATA_DIR if DATA_DIR else "not found - the next cell will rebuild it")

### If the processed data is missing

Seeing `Data : not found` above is normal, not an error.

Both Colab and Kaggle give every notebook its own separate machine. Notebook 02 wrote its results to that machine's temporary disk, so this notebook cannot see them — and neither can any future session.

Rather than shuttling files between sessions, the cell below simply rebuilds the cleaned dataset from the raw images. It repeats exactly what Notebook 02 did: the same three-pass duplicate removal, the same conversion to 224×224 colour images, and the same random seed — so the folds come out identical and results stay comparable.

It takes about five minutes, mostly re-downloading the raw images. If the data *is* already present, the cell does nothing and finishes instantly.

In [ ]:
if DATA_DIR is None:
    import hashlib
    import os
    import shutil
    from collections import Counter

    from sklearn.model_selection import StratifiedKFold

    print("Processed data not found - rebuilding it from the raw dataset.\n")

    # --- locate the raw dataset ---
    ds2 = "/kaggle/input/multi-class-fabric-defect-detection-dataset/Dataset"
    if not os.path.exists(ds2):
        import kagglehub
        ds2 = os.path.join(
            kagglehub.dataset_download("ziya07/multi-class-fabric-defect-detection-dataset"),
            "Dataset",
        )
    raw_classes = sorted(os.listdir(ds2))
    print("Raw dataset:", ds2)

    def dhash(path, size=16):
        with Image.open(path) as img:
            small = img.convert("L").resize((size + 1, size), Image.BILINEAR)
        px = list(small.getdata())
        bits = []
        for row in range(size):
            for col in range(size):
                left = px[row * (size + 1) + col]
                right = px[row * (size + 1) + col + 1]
                bits.append("1" if left > right else "0")
        return "".join(bits)

    # --- pass 1: drop the processed derivatives ---
    candidates = [
        (cls, p)
        for cls in raw_classes
        for p in sorted((Path(ds2) / cls).glob("*"))
        if "_processed" not in p.name
    ]
    print(f"After removing _processed     : {len(candidates)}")

    # --- pass 2: drop exact duplicates ---
    seen_md5, after_exact = set(), []
    for cls, p in candidates:
        h = hashlib.md5(p.read_bytes()).hexdigest()
        if h in seen_md5:
            continue
        seen_md5.add(h)
        after_exact.append((cls, p))
    print(f"After removing exact copies   : {len(after_exact)}")

    # --- pass 3: drop near-duplicates ---
    seen_dhash, keep = set(), []
    for cls, p in after_exact:
        key = (cls, dhash(p))
        if key in seen_dhash:
            continue
        seen_dhash.add(key)
        keep.append((cls, p))
    print(f"After removing near-duplicates: {len(keep)}\n")

    # --- assign folds (same seed as Notebook 02, so the splits match) ---
    labels = [cls for cls, _ in keep]
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_of = {}
    for fold_idx, (_, test_idx) in enumerate(skf.split(range(len(keep)), labels)):
        for i in test_idx:
            fold_of[i] = fold_idx

    # --- resize and save ---
    OUT = Path("/kaggle/working/processed") if Path("/kaggle/working").exists() \
        else Path("data/processed")
    if OUT.exists():
        shutil.rmtree(OUT)
    (OUT / "images").mkdir(parents=True)

    rows = []
    for i, (cls, p) in enumerate(keep):
        cls_dir = OUT / "images" / cls
        cls_dir.mkdir(exist_ok=True)
        name = f"{i:05d}.jpg"
        with Image.open(p) as im:
            im.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR).save(
                cls_dir / name, quality=95)
        rows.append({
            "filepath": str(Path("images") / cls / name),
            "class": cls,
            "fold": fold_of[i],
            "source_file": p.name,
        })
    pd.DataFrame(rows).to_csv(OUT / "manifest.csv", index=False)

    # --- class weights ---
    counts = Counter(labels)
    total_imgs = len(labels)
    weights = {c: total_imgs / (len(raw_classes) * counts[c]) for c in raw_classes}
    with open(OUT / "class_weights.json", "w") as f:
        json.dump(weights, f, indent=2)

    DATA_DIR = OUT
    print("Rebuilt at:", DATA_DIR.resolve())
else:
    print("Using the processed data already found at:", DATA_DIR)

In [ ]:
manifest = pd.read_csv(DATA_DIR / "manifest.csv")
class_weights_raw = json.load(open(DATA_DIR / "class_weights.json"))
class_names = sorted(manifest["class"].unique())
n_classes = len(class_names)
cls_to_idx = {c: i for i, c in enumerate(class_names)}

print(f"Images : {len(manifest)}")
print(f"Classes: {n_classes}")
print(f"Folds  : {manifest['fold'].nunique()}")
print()
print(manifest["class"].value_counts().to_string())

**Observations:** _(fill in — how many images loaded, which device is being used, and does the class distribution match Notebook 02?)_

## 2. Loading and Augmenting the Images

**What:** Define how images are read and randomly altered during training.

**Why augment at all?** We have 2,737 images and only about 32 in the smallest class. A network with hundreds of thousands of parameters can simply memorise a set that small — scoring perfectly on training images while failing on anything new. Augmentation shows the model a slightly different version of each image every time it sees it: flipped, rotated a little, brighter or dimmer. The model can no longer memorise exact pictures, so it is pushed towards learning what a defect actually looks like.

**One augmentation we must not use.** Rotating an image by 90° would turn a *vertical* defect into a *horizontal* one — while the label still says "Vertical". We would be generating deliberately wrong training data for the two classes that are already the smallest and hardest.

Flips are safe, which is worth being clear about: a vertical line flipped left-to-right is still a vertical line. Only rotation changes direction, so rotations stay small (±10°).

**Why change the colours.** Notebook 01 found the dataset was assembled from several collections with different cameras and lighting. Randomly shifting brightness, contrast and colour makes those source differences unreliable as a clue, pushing the model to focus on fabric texture instead. Occasionally converting an image to grayscale does the same job more forcefully.

**Why normalise.** We subtract a fixed mean and divide by a fixed standard deviation so that pixel values are centred around zero, which helps training converge. We use the standard ImageNet values so this pipeline carries over unchanged to Notebook 04's pretrained model.

Note that augmentation applies **only during training**. When measuring performance we use the plain image, because the score should reflect real data rather than randomly distorted data.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(10),                       # small only - see note above
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class FabricDataset(Dataset):
    def __init__(self, df, root, transform):
        self.df = df.reset_index(drop=True)
        self.root = Path(root)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.root / row["filepath"]).convert("RGB")
        return self.transform(img), cls_to_idx[row["class"]]


print("Transforms ready.")
print("Train pipeline:", len(train_tf.transforms), "steps")
print("Eval pipeline :", len(eval_tf.transforms), "steps")

In [ ]:
# Show one image with several random augmentations applied
sample_row = manifest.sample(1, random_state=SEED).iloc[0]
sample_img = Image.open(DATA_DIR / sample_row["filepath"]).convert("RGB")

fig, axes = plt.subplots(1, 6, figsize=(20, 4))
axes[0].imshow(sample_img)
axes[0].set_title(f"original\n({sample_row['class']})", fontsize=10)
axes[0].axis("off")

for ax in axes[1:]:
    aug = train_tf(sample_img)
    shown = aug * torch.tensor(IMAGENET_STD).view(3, 1, 1) + torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    ax.imshow(shown.permute(1, 2, 0).clamp(0, 1))
    ax.set_title("augmented", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

**Observations:** _(fill in — is the defect still clearly visible in every augmented version? Has any augmentation changed the image so much that the label would no longer be correct?)_

## 3. The Baseline CNN

**What:** Build a small convolutional network from scratch.

**Why this design?** The network stacks four blocks, each containing:

- a **convolution**, which scans the image for small patterns — early blocks find edges and texture, later blocks combine those into larger shapes like a hole or a stitch line
- **batch normalisation**, which keeps the numbers flowing through the network in a stable range and makes training much faster
- a **ReLU**, which lets the network model complicated relationships rather than just straight-line ones
- **max pooling**, which halves the image size, so later blocks see a wider area of the original picture

Channels double at each block (32 → 64 → 128 → 256): as the picture gets smaller, the network can afford to look for more distinct kinds of pattern.

At the end, **global average pooling** collapses each channel to a single number, then **dropout** randomly switches off half the values during training. Dropout is important here — with only 2,737 images, it stops the network leaning too heavily on any one feature and helps it generalise.

This is deliberately a *modest* network. It is not meant to be the best model we can build; it is meant to be an honest measure of what an ordinary CNN achieves on this data without any outside knowledge.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_classes, dropout=0.5):
        super().__init__()

        def block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(
            block(3, 32),      # 224 -> 112
            block(32, 64),     # 112 -> 56
            block(64, 128),    # 56  -> 28
            block(128, 256),   # 28  -> 14
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))


model = SimpleCNN(n_classes)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTrainable parameters: {n_params:,}")

**Observations:** _(fill in — how many parameters does the network have, and how does that compare with the 2,737 training images available?)_

## 4. Training One Fold

**What:** Define the routine that trains the model once and measures it on held-out images.

**Why weight the loss?** Notebook 02 calculated a weight for each class based on how rare it is. We pass those weights into the loss function so that misclassifying a "Vertical" image costs far more than misclassifying a "defect free" one. Without this, the model quickly discovers it can score 60% by always answering "defect free", and training would push it straight towards that dead end.

**Why a fixed number of epochs instead of stopping early?** A common technique is to watch performance on held-out data and stop when it stops improving. We deliberately do not do that here. The held-out fold is what we use to *report* our score, so using it to decide when to stop would mean peeking at the answer — the reported number would look better than the model truly deserves.

Instead we train for a fixed 25 epochs every time and report the result at the end. We still record the held-out score after each epoch, but only to draw a learning curve and check that training behaved sensibly.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, all_preds, all_true = 0.0, [], []
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            all_preds.append(out.argmax(1).cpu())
            all_true.append(y.cpu())

    preds = torch.cat(all_preds).numpy()
    true = torch.cat(all_true).numpy()
    return total_loss / len(loader.dataset), preds, true


def train_fold(fold, verbose=True):
    train_df = manifest[manifest["fold"] != fold]
    test_df = manifest[manifest["fold"] == fold]

    train_loader = DataLoader(
        FabricDataset(train_df, DATA_DIR, train_tf),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=False)
    test_loader = DataLoader(
        FabricDataset(test_df, DATA_DIR, eval_tf),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    torch.manual_seed(SEED + fold)
    model = SimpleCNN(n_classes).to(device)

    weights = torch.tensor([class_weights_raw[c] for c in class_names],
                           dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    history = []
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_pred, tr_true = run_epoch(model, train_loader, criterion, optimizer)
        te_loss, te_pred, te_true = run_epoch(model, test_loader, criterion)

        tr_f1 = f1_score(tr_true, tr_pred, average="macro", zero_division=0)
        te_f1 = f1_score(te_true, te_pred, average="macro", zero_division=0)
        history.append({"epoch": epoch, "train_loss": tr_loss, "test_loss": te_loss,
                        "train_f1": tr_f1, "test_f1": te_f1})

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f"  epoch {epoch:2d}/{EPOCHS}  "
                  f"train loss {tr_loss:.3f} f1 {tr_f1:.3f}  |  "
                  f"held-out loss {te_loss:.3f} f1 {te_f1:.3f}")

    return model, pd.DataFrame(history), te_pred, te_true, test_df.index.values


print("Training routine defined.")

## 5. Train Across All Five Folds

**What:** Train the network five times, each time holding out a different fold for testing.

**Why:** Notebook 02 split the data into five folds precisely for this. Each image is held out exactly once, so at the end we have a prediction for **every image in the dataset**, each made by a model that never saw that image during training.

This matters most for the rare classes. A single 15% test split would leave "Vertical" with about 5 test images, and a score based on 5 images swings wildly by chance. Across five folds we get predictions for all ~32, which is enough for the number to mean something.

Training five models takes roughly five times as long as training one. Expect several minutes per fold on a GPU, considerably longer on a laptop.

In [ ]:
oof_pred = np.zeros(len(manifest), dtype=int)
oof_true = np.zeros(len(manifest), dtype=int)
histories, fold_scores = [], []

start = time.time()
for fold in sorted(manifest["fold"].unique()):
    print(f"\nFold {fold}")
    model, hist, pred, true, idx = train_fold(fold)

    oof_pred[idx] = pred
    oof_true[idx] = true
    hist["fold"] = fold
    histories.append(hist)

    f1 = f1_score(true, pred, average="macro", zero_division=0)
    acc = (pred == true).mean()
    fold_scores.append({"fold": fold, "macro_f1": f1, "accuracy": acc})
    print(f"  -> macro F1 {f1:.3f} | accuracy {acc:.3f}")

history_df = pd.concat(histories, ignore_index=True)
scores_df = pd.DataFrame(fold_scores)

print(f"\nTotal training time: {(time.time() - start) / 60:.1f} minutes")
print()
print(scores_df.to_string(index=False))
print(f"\nMean macro F1: {scores_df['macro_f1'].mean():.3f} "
      f"(+/- {scores_df['macro_f1'].std():.3f})")

**Observations:** _(fill in — how much did macro F1 vary between folds? A large spread means the result depends heavily on which images were held out.)_

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for fold, g in history_df.groupby("fold"):
    axes[0].plot(g["epoch"], g["train_loss"], alpha=0.7, label=f"fold {fold}")
    axes[1].plot(g["epoch"], g["test_f1"], alpha=0.7, label=f"fold {fold}")

axes[0].set_title("Training loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")
axes[1].set_title("Held-out macro F1")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("macro F1")
for ax in axes:
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Observations:** _(fill in — did the loss keep falling? Did the held-out score flatten out, keep improving, or start getting worse? A held-out score that falls while training loss keeps dropping means the model is memorising the training images.)_

## 6. Results

**What:** Score the predictions collected across all five folds.

**Why report it this way?** Every image now has a prediction from a model that never saw it during training, so these numbers reflect genuinely unseen data.

We report three things:

- **Macro F1** — our headline. It averages the score across all nine classes equally, so failing on the rare defects cannot be hidden by succeeding on the common ones.
- **Accuracy** — included only for comparison against the majority-class reference. On its own it is misleading here.
- **Recall for each class** — of all the images that really were, say, "Vertical", what fraction did we correctly identify? In a factory this is the number that matters most: a missed defect ships to the customer.

In [ ]:
macro_f1 = f1_score(oof_true, oof_pred, average="macro", zero_division=0)
accuracy = (oof_pred == oof_true).mean()

print(f"Overall macro F1 : {macro_f1:.3f}")
print(f"Overall accuracy : {accuracy:.3f}")
print()
print(classification_report(oof_true, oof_pred, target_names=class_names,
                            digits=3, zero_division=0))

**Observations:** _(fill in — which classes does the model handle well, and which does it struggle with? Do the rare classes ("Vertical", "horizontal") perform noticeably worse?)_

In [ ]:
recalls = recall_score(oof_true, oof_pred, average=None, zero_division=0)
counts = manifest["class"].value_counts()

rare = pd.DataFrame({
    "class": class_names,
    "images": [counts[c] for c in class_names],
    "recall": recalls,
}).sort_values("images")

print("Recall by class, rarest first:")
print(rare.to_string(index=False))

**Observations:** _(fill in — is there a clear pattern between how rare a class is and how well the model detects it?)_

In [ ]:
cm = confusion_matrix(oof_true, oof_pred)
cm_pct = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(n_classes)); ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticks(range(n_classes)); ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Baseline CNN — confusion matrix (row %)")

for i in range(n_classes):
    for j in range(n_classes):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                    color="white" if cm_pct[i, j] > 0.5 else "black")

plt.colorbar(im, label="fraction of actual class")
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"{MODEL_NAME}_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

**Observations:** _(fill in — which classes get confused with each other? Are mistakes concentrated on "defect free", meaning defects are being missed, or between similar defect types?)_

## 7. Did It Beat the Majority-Class Reference?

**What:** Compare the baseline CNN against the majority-class reference — always answering "defect free".

**Why this comparison is the point of the notebook.** Notebook 01 measured the majority-class reference at 60.4% accuracy and 0.084 macro F1. A CNN that scores, say, 65% accuracy might look like a success — but if its macro F1 is near 0.084, it has essentially learned to copy the majority class and is worthless as a defect detector.

Macro F1 is the honest comparison, because it only improves if the model genuinely handles the rare classes better.

In [ ]:
majority_class = manifest["class"].value_counts().idxmax()
p = manifest["class"].value_counts().max() / len(manifest)
ref_acc = p
ref_f1 = (2 * p / (p + 1)) / n_classes

comparison = pd.DataFrame([
    {"model": f"Majority-class reference ({majority_class})", "accuracy": ref_acc, "macro_f1": ref_f1},
    {"model": "Baseline CNN (trained from scratch)", "accuracy": accuracy, "macro_f1": macro_f1},
])
comparison["macro_f1_gain"] = comparison["macro_f1"] - ref_f1

print(comparison.to_string(index=False))

**Observations:** _(fill in — by how much did macro F1 improve over doing nothing? Did accuracy improve by a similar amount, or much less? A big accuracy gain with a small macro F1 gain means the model is mostly just predicting the majority class.)_

## 8. Save the Results

**What:** Write the scores, per-epoch history, and predictions to disk.

**Why:** Notebook 04 needs these numbers to compare ResNet18 against this baseline CNN. Saving the per-image predictions also means we can revisit specific mistakes later without retraining anything.

In [ ]:
scores_df.to_csv(RESULTS_DIR / f"{MODEL_NAME}_fold_scores.csv", index=False)
history_df.to_csv(RESULTS_DIR / f"{MODEL_NAME}_history.csv", index=False)

pred_df = manifest.copy()
pred_df["predicted"] = [class_names[i] for i in oof_pred]
pred_df["correct"] = pred_df["predicted"] == pred_df["class"]
pred_df.to_csv(RESULTS_DIR / f"{MODEL_NAME}_predictions.csv", index=False)

summary = {
    "model": "SimpleCNN (from scratch)",
    "n_images": int(len(manifest)),
    "n_classes": int(n_classes),
    "n_folds": int(manifest["fold"].nunique()),
    "epochs": EPOCHS,
    "parameters": int(n_params),
    "macro_f1": float(macro_f1),
    "accuracy": float(accuracy),
    "per_class_recall": {c: float(r) for c, r in zip(class_names, recalls)},
    "reference_macro_f1": float(ref_f1),
    "reference_accuracy": float(ref_acc),
}
with open(RESULTS_DIR / f"{MODEL_NAME}_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved to", RESULTS_DIR.resolve())
for f in sorted(RESULTS_DIR.iterdir()):
    print(f"   {f.name:38s} {f.stat().st_size / 1024:8.1f} KB")

## 9. Summary

### What this notebook did

1. Loaded the 2,737 cleaned images from Notebook 02, along with their fold assignments and class weights.
2. Set up augmentation that avoids 90° rotations, since those would turn a "Vertical" defect into a "horizontal" one and create wrongly labelled training data.
3. Built a small four-block CNN from scratch — no outside knowledge, trained only on this dataset.
4. Trained it five times using cross-validation, so every image was tested by a model that had never seen it.
5. Reported macro F1 and per-class recall, and compared both against the majority-class reference.

### Files produced

Each model writes into its own folder under `outputs/`, so Notebook 04's ResNet18 results sit beside these rather than mixed in with them:

```
outputs/
│
├── baseline_cnn/
│   ├── baseline_cnn_summary.json
│   ├── baseline_cnn_fold_scores.csv
│   ├── baseline_cnn_history.csv
│   ├── baseline_cnn_predictions.csv
│   └── baseline_cnn_confusion_matrix.png
│
└── resnet18/            ← created by Notebook 04
    └── ...
```

| File | Contents |
|---|---|
| `baseline_cnn_summary.json` | Headline scores and per-class recall |
| `baseline_cnn_fold_scores.csv` | Macro F1 and accuracy for each fold |
| `baseline_cnn_history.csv` | Loss and F1 after every epoch |
| `baseline_cnn_predictions.csv` | The prediction for every image |
| `baseline_cnn_confusion_matrix.png` | Confusion matrix figure |

Keeping the model name in both the folder and the filenames is deliberate: it means a file stays identifiable if it is copied into a report or attached to an email, away from its folder.

### Decisions worth defending

- **No 90° rotations in augmentation.** They would relabel "Vertical" as "horizontal" while leaving the label untouched, damaging the two smallest and hardest classes.
- **Weighted loss rather than duplicating rare images.** With only ~32 "Vertical" images, copying them would teach the model to memorise those 32 pictures.
- **A fixed 25 epochs rather than stopping early.** Stopping early would require watching the held-out fold, and that fold is what we report our score on — deciding when to stop by looking at it would flatter the result.
- **Cross-validation rather than one split.** It gives roughly 32 test predictions for "Vertical" instead of about 5, which is the difference between a measurement and a guess.

### A note on saving these results

On Kaggle, the files above are written to `/kaggle/working/outputs/` and are lost when the session ends unless you **Save Version**. Running locally, they go to `outputs/` — which is currently listed in `.gitignore`, so they will not reach GitHub. If you want the confusion matrix in your README or final report, either copy it into `assets/` or remove `outputs/` from `.gitignore`.

### Next: Notebook 04

Train **ResNet18** using transfer learning, with the same folds, the same augmentation and the same metrics, so the comparison is fair. The question it answers: does starting from a network already trained on millions of photographs beat learning from these 2,737 fabric images alone?